# Data Cleaning
**This notebook prepares Flatiron Health CSV files for patients with metastatic renal cell cancer treated with first-line combination immunotherapy or single-agent antiangiogenic therapy. Those with clear cell or NOS histology are selected. Each CSV is cleaned using the flatiron_cleaner package. The cleaned dataframes are then merged into a single dataset, which will be used for survival analysis.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorRenal, merge_dataframes

## Import data

In [2]:
df = pd.read_csv('../outputs/ioio_tki_index.csv')

In [3]:
df.head(5)

,PatientID,LineName,StartDate
0,F50AB9E183B8B,ioio,2020-04-02
1,F0DAF71F2552A,ioio,2019-04-25
2,F7F5EFF0FA2B5,ioio,2019-11-21
3,FA68EFBE8B2B0,ioio,2021-09-21
4,FC9B9EAE7E830,ioio,2018-07-30


In [4]:
df.shape

(4831, 3)

In [5]:
ids = df.PatientID.to_list()

## Clean CSV files 

In [6]:
# Initialize class 
processor = DataProcessorRenal()

### Process Enhanced_MetastaticRCC.csv

In [7]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_MetastaticRCC.csv',
                                         index_date_df = df,
                                         index_date_column = 'StartDate',
                                         drop_dates = False)

2026-03-20 22:16:23,069 - INFO - Successfully read Enhanced_MetastaticRCC.csv file with shape: (10766, 10) and unique PatientIDs: 10766
2026-03-20 22:16:23,073 - INFO - Successfully filtered Enhanced_MetastaticRCC.csv file with shape: (4831, 11) and unique PatientIDs: 4831
2026-03-20 22:16:23,082 - INFO - Successfully processed Enhanced_MetastaticCRC.csv file with final shape: (4831, 13) and unique PatientIDs: 4831


In [8]:
enhanced_df = enhanced_df.copy()

In [9]:
# remove patients with missing MetDiagnosisDate
enhanced_df = enhanced_df.query('MetDiagnosisDate.notna()')
enhanced_df.shape

(4813, 13)

In [10]:
enhanced_df['days_met_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['MetDiagnosisDate']).dt.days
enhanced_df['days_met_to_treatment'] = np.where(enhanced_df['days_met_to_treatment'] < 0 , 0, enhanced_df['days_met_to_treatment'])

enhanced_df['days_diag_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['DiagnosisDate']).dt.days
enhanced_df['days_diag_to_treatment'] = np.where(enhanced_df['days_diag_to_treatment'] < 0 , 0, enhanced_df['days_diag_to_treatment'])

enhanced_df['days_to_treatment_before_30d'] = np.where(enhanced_df['days_met_to_treatment'] < 30, 1, 0)
enhanced_df['days_to_treatment_before_1y'] = np.where(enhanced_df['days_diag_to_treatment'] < 365, 1, 0)

In [11]:
enhanced_df['days_diagnosis_to_met'] = np.where(enhanced_df['days_diagnosis_to_met'] < 0 , 0, enhanced_df['days_diagnosis_to_met'])
enhanced_df['days_diagnosis_to_met'] = enhanced_df['days_diagnosis_to_met'].fillna(0)

In [12]:
enhanced_df['met_diagnosis_year'] = enhanced_df['met_diagnosis_year'].astype(int)

In [13]:
enhanced_df.SmokingStatus.value_counts(dropna = False)

SmokingStatus
History of smoking        2791
No history of smoking     1983
Unknown/Not documented      39
Name: count, dtype: int64

In [14]:
enhanced_df['SmokingStatus'] = enhanced_df['SmokingStatus'].map({
    'History of smoking': 1,
    'No history of smoking': 0,
    'Unknown/Not documented': 1
})

In [15]:
enhanced_df.StageFourDetail.value_counts(dropna = False)

StageFourDetail
M1         2461
NaN        2306
T4/M0        43
Unknown       3
Name: count, dtype: int64

In [16]:
enhanced_df.NephrectomyType_mod.value_counts(dropna = False)

NephrectomyType_mod
Radical nephrectomy    2748
Unknown                1812
Partial nephrectomy     250
Other                     3
Name: count, dtype: int64

In [17]:
enhanced_df['NephrectomyType_mod'] = enhanced_df['NephrectomyType_mod'].map(
    lambda x: 'other/unknown' if x in ['Unknown', 'Other'] else x
)

enhanced_df['NephrectomyType_mod'] = enhanced_df['NephrectomyType_mod'].astype('category')

In [18]:
enhanced_df.Histology.value_counts()

Histology
Clear Cell         3428
RCC, NOS            943
Papillary           314
Chromophobe          68
Other                42
Translocation         7
Renal Medullary       3
Name: count, dtype: int64

In [19]:
enhanced_df = enhanced_df.query('Histology == "Clear Cell" or Histology == "RCC, NOS"')
enhanced_df.shape

(4371, 17)

In [20]:
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'MetDiagnosisDate',
                                          'NephrectomyDate',
                                          'imported_StartDate'])

### Process Demographics.csv 

In [21]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = df,
                                                 index_date_column = 'StartDate')

2026-03-20 22:16:23,138 - INFO - Successfully read Demographics.csv file with shape: (10766, 6) and unique PatientIDs: 10766
2026-03-20 22:16:23,144 - WARNING - Found 1 ages outside valid range (18-120)
2026-03-20 22:16:23,148 - INFO - Successfully processed Demographics.csv file with final shape: (4831, 6) and unique PatientIDs: 4831


In [22]:
demographics_df = demographics_df.copy()

In [23]:
demographics_df = demographics_df.query('age >= 18 and age <= 120', engine = 'python')

In [24]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M    3431
F    1399
Name: count, dtype: int64

In [25]:
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [26]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_MetRCCBiomarkers.csv

In [27]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_MetRCCBiomarkers.csv',
                                             index_date_df = df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2026-03-20 22:16:23,167 - INFO - Successfully read Enhanced_MetRCCBiomarkers.csv file with shape: (878, 17) and unique PatientIDs: 692
2026-03-20 22:16:23,171 - INFO - Successfully merged Enhanced_MetRCCBiomarkers.csv df with index_date_df resulting in shape: (378, 18) and unique PatientIDs: 304
2026-03-20 22:16:23,184 - INFO - Successfully processed Enhanced_MetRCCBiomarkers.csv file with final shape: (4831, 3) and unique PatientIDs: 4831


In [28]:
biomarkers_df.PDL1_percent_staining.value_counts(dropna = False)

PDL1_percent_staining
NaN          4814
5% - 9%         5
10% - 19%       2
50% - 59%       2
80% - 89%       2
1%              1
30% - 39%       1
60% - 69%       1
70% - 79%       1
90% - 99%       1
100%            1
0%              0
< 1%            0
2% - 4%         0
20% - 29%       0
40% - 49%       0
Name: count, dtype: int64

In [29]:
def map_pdl1(value):
    if pd.isna(value):  # leave missing as is
        return value
    elif value in ['0%', '< 1%']:
        return '0%'
    else:
        return '>=1%'

biomarkers_df['PDL1_binary'] = biomarkers_df['PDL1_percent_staining'].apply(map_pdl1)

In [30]:
biomarkers_df.PDL1_binary.value_counts(dropna = False)

PDL1_binary
NaN     4814
>=1%      17
Name: count, dtype: int64

In [31]:
biomarkers_df['PDL1_status'] = biomarkers_df['PDL1_status'].fillna('unknown')

In [32]:
biomarkers_df = biomarkers_df.drop(columns = ['PDL1_percent_staining'])

### Process ECOG.csv

In [33]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-03-20 22:16:23,247 - INFO - Successfully read ECOG.csv file with shape: (141710, 4) and unique PatientIDs: 7722
2026-03-20 22:16:23,276 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (80875, 5) and unique PatientIDs: 3603
2026-03-20 22:16:23,327 - INFO - Successfully processed ECOG.csv file with final shape: (4831, 3) and unique PatientIDs: 4831


In [34]:
ecog_df.ecog_index.value_counts(dropna = False)

ecog_index
NaN    2236
0      1093
1      1021
2       371
3        97
4        13
Name: count, dtype: int64

In [35]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 0 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(0)

In [36]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [37]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2026-03-20 22:16:25,178 - INFO - Successfully read Vitals.csv file with shape: (2216004, 16) and unique PatientIDs: 10743
2026-03-20 22:16:26,178 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (1099686, 17) and unique PatientIDs: 4828
2026-03-20 22:16:26,748 - INFO - Successfully processed Vitals.csv file with final shape: (4831, 8) and unique PatientIDs: 4831


### Process Lab.csv

In [38]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 additional_loinc_mappings = {'neutrophil': ['26499-4', '751-8', '753-4']},
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-03-20 22:16:35,253 - INFO - Successfully read Lab.csv file with shape: (7341289, 17) and unique PatientIDs: 10244
2026-03-20 22:16:37,409 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (3845426, 18) and unique PatientIDs: 4684
2026-03-20 22:16:44,896 - INFO - Successfully processed Lab.csv file with final shape: (4831, 81) and unique PatientIDs: 4831


### Process MedicationAdministration.csv

In [39]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2026-03-20 22:16:45,414 - INFO - Successfully read MedicationAdministration.csv file with shape: (407234, 11) and unique PatientIDs: 7413
2026-03-20 22:16:45,540 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (162206, 12) and unique PatientIDs: 3606
2026-03-20 22:16:45,570 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (4831, 9) and unique PatientIDs: 4831


### Process Diagnosis.csv

In [40]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2026-03-20 22:16:45,786 - INFO - Successfully read Diagnosis.csv file with shape: (342031, 6) and unique PatientIDs: 10766
2026-03-20 22:16:45,848 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (140375, 7) and unique PatientIDs: 4831
2026-03-20 22:16:46,263 - INFO - Successfully processed Diagnosis.csv file with final shape: (4831, 40) and unique PatientIDs: 4831


In [41]:
diagnosis_df['gi_met_combined'] = (
    diagnosis_df['adrenal_met'] | diagnosis_df['peritoneum_met'] | diagnosis_df['gi_met']
)

diagnosis_df = diagnosis_df.drop(columns = ['adrenal_met', 'peritoneum_met', 'gi_met'])

### Process Enhanced_Mortality_V2.csv

In [42]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_MetRCCBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_MetRCC_Orals.csv',
                                           progression_path = '../data/Enhanced_MetRCCProgression.csv',
                                           drop_dates = False)

2026-03-20 22:16:46,278 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (6345, 2) and unique PatientIDs: 6345
2026-03-20 22:16:46,287 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (4831, 3) and unique PatientIDs: 4831
2026-03-20 22:16:46,577 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-03-20 22:16:46,582 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (4831, 6) and unique PatientIDs: 4831. There are 0 out of 4831 patients with missing duration values


In [43]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [44]:
mortality_df = mortality_df.query('duration >= 0')

### Process Insurance.csv

In [45]:
insurance_df = processor.process_insurance(file_path = '../data/Insurance.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0,
                                           missing_date_strategy = 'liberal')

2026-03-20 22:16:46,615 - INFO - Successfully read Insurance.csv file with shape: (24228, 5) and unique PatientIDs: 10043
2026-03-20 22:16:46,628 - INFO - Successfully merged Insurance.csv df with index_date_df resulting in shape: (11576, 6) and unique PatientIDs: 4509
2026-03-20 22:16:46,642 - INFO - Successfully processed Insurance.csv file with final shape: (4831, 5) and unique PatientIDs: 4831


## Merge dataframes

In [46]:
ioio_tki_features_df = merge_dataframes(enhanced_df,
                                        demographics_df,
                                        biomarkers_df,
                                        ecog_df,
                                        vitals_df,
                                        labs_df,
                                        medications_df,
                                        diagnosis_df,
                                        mortality_df, 
                                        insurance_df, 
                                        merge_type = 'inner')

2026-03-20 22:16:46,645 - INFO - Anticipated number of merges: 9
2026-03-20 22:16:46,646 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 161
2026-03-20 22:16:46,647 - INFO - Dataset 1 shape: (4371, 13), unique PatientIDs: 4371
2026-03-20 22:16:46,648 - INFO - Dataset 2 shape: (4830, 6), unique PatientIDs: 4830
2026-03-20 22:16:46,649 - INFO - Dataset 3 shape: (4831, 3), unique PatientIDs: 4831
2026-03-20 22:16:46,650 - INFO - Dataset 4 shape: (4831, 4), unique PatientIDs: 4831
2026-03-20 22:16:46,650 - INFO - Dataset 5 shape: (4831, 8), unique PatientIDs: 4831
2026-03-20 22:16:46,651 - INFO - Dataset 6 shape: (4831, 81), unique PatientIDs: 4831
2026-03-20 22:16:46,652 - INFO - Dataset 7 shape: (4831, 9), unique PatientIDs: 4831
2026-03-20 22:16:46,652 - INFO - Dataset 8 shape: (4831, 38), unique PatientIDs: 4831
2026-03-20 22:16:46,653 - INFO - Dataset 9 shape: (4827, 3), unique PatientIDs: 4827
2026-03-20 22:16:46,654 - 

In [47]:
ioio_tki_features_df.shape

(4366, 161)

In [48]:
ioio_tki_features_df.head(2)

,PatientID,GroupStage,StageFourDetail,Histology,SmokingStatus,Nephrectomy_mod,NephrectomyType_mod,days_diagnosis_to_met,met_diagnosis_year,days_met_to_treatment,...,bone_met,brain_met,other_met,gi_met_combined,event,duration,commercial,medicaid,medicare,other_insurance
0,F42ACFD609CE6,I - III,NaN,Clear Cell,1,1,Partial nephrectomy,1092.0,2015,0,...,0,0,0,0,0,215.0,1,0,1,0
1,F2A697A9BC9E5,IV,M1,"RCC, NOS",1,0,other/unknown,0.0,2016,20,...,0,0,0,0,1,955.0,0,0,1,0


## Export dataframe

In [49]:
ioio_tki_features_df.to_csv('../outputs/ioio_tki_features.csv', index = False)

In [50]:
# Save dtypes
ioio_tki_features_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/ioio_tki_features_dtypes.csv')